In [ ]:
import os
import cv2
import pandas as pd

# ============================================================
# Select Dataset for train and augmented change path manually
# ============================================================

dataset = input("Which dataset do you want to label? (train / valid / test/train_2): ").strip().lower()

BASE_FOLDER = r"../data_set"

if dataset == "train":
    IMAGE_FOLDER = os.path.join(BASE_FOLDER, "train")
    CSV_FILE = os.path.join(BASE_FOLDER, "train.csv")

elif dataset in ["valid", "validation", "val"]:
    IMAGE_FOLDER = os.path.join(BASE_FOLDER, "valid")
    CSV_FILE = os.path.join(BASE_FOLDER, "valid.csv")

elif dataset == "test":
    IMAGE_FOLDER = os.path.join(BASE_FOLDER, "test")
    CSV_FILE = os.path.join(BASE_FOLDER, "test.csv")
elif dataset == "train_2":
    IMAGE_FOLDER = os.path.join(BASE_FOLDER, "train_2")
    CSV_FILE = os.path.join(BASE_FOLDER, "train_2.csv")

else:
    print("Invalid dataset.")
    exit()

SUPPORTED_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp")

# ============================================================
# Helper function to get base name
# ============================================================
def get_original_stem(filename):
    """ Extracts base file string before '_crop_X' or '_no_detection' """
    stem = os.path.splitext(filename)[0]
    if "_no_detection" in stem:
        return stem.split("_no_detection")[0]
    if "_crop_" in stem:
        return stem.split("_crop_")[0]
    return stem

# ============================================================
# Resume Support & Stem Tracking
# ============================================================

if os.path.exists(CSV_FILE):
    df = pd.read_csv(CSV_FILE, dtype={"label": str})
    labeled_images = set(df["image"].astype(str))
    # Pre-calculate stems of already completed images for fast, exact matching
    labeled_stems = {get_original_stem(img) for img in labeled_images}
else:
    df = pd.DataFrame(columns=["image", "label"])
    labeled_images = set()
    labeled_stems = set()

# ============================================================
# Image List
# ============================================================

images = sorted([
    img for img in os.listdir(IMAGE_FOLDER)
    if img.lower().endswith(SUPPORTED_EXTENSIONS)
])

print(f"\nDataset : {dataset}")
print(f"Total Images found in folder : {len(images)}")
print(f"Already Labeled in CSV       : {len(labeled_images)}")

# ============================================================
# Labeling Loop
# ============================================================

for index, image_name in enumerate(images):

    # FIX: Exact set matching prevents substring partial match bugs
    base_stem = get_original_stem(image_name)
    if image_name in labeled_images or base_stem in labeled_stems:
        continue

    image_path = os.path.join(IMAGE_FOLDER, image_name)
    img = cv2.imread(image_path)

    if img is None:
        print(f"Could not read {image_name}")
        continue

    # Resize only for display
    h, w = img.shape[:2]
    scale = min(900 / w, 400 / h)
    display = cv2.resize(img, (int(w * scale), int(h * scale)))
    cv2.namedWindow("Meter Labeling Tool", cv2.WINDOW_NORMAL)
    cv2.imshow("Meter Labeling Tool", display)
    cv2.resizeWindow("Meter Labeling Tool", 900, 400)

    # Force the image to appear before asking for input
    cv2.waitKey(100)

    print(f"\n[{index+1}/{len(images)}] {image_name}")
    while True:

        label = input("Enter 5-digit reading (q = quit, s = skip, d = delete): ").strip()

        if label.lower() == "q":
            cv2.destroyAllWindows()
            df.to_csv(CSV_FILE, index=False)
            print("\nProgress Saved.")
            exit()

        if label.lower() == "s":
            break

        # -----------------------------
        # NEW FEATURE: Delete Image
        # -----------------------------
        if label.lower() in ["d", "del", "delete"]:
            cv2.destroyWindow("Meter Labeling Tool") # Close window to avoid OS file locks
            try:
                os.remove(image_path)
                print(f"🔥 Deleted file: {image_name}")
            except Exception as e:
                print(f"Error deleting file: {e}")
            break

        if len(label) == 5 and label.isdigit():
            df.loc[len(df)] = [image_name, label]
            df.to_csv(CSV_FILE, index=False)
            # Make sure we add it to our skips if we keep moving forward
            labeled_images.add(image_name)
            labeled_stems.add(base_stem)
            break

        print("Invalid label! Please enter exactly 5 digits.")

cv2.destroyAllWindows()
print("\n===================================")
print("All pending images have been processed!")
print("CSV saved to:")
print(CSV_FILE)
print("===================================")

Invalid dataset.


NameError: name 'CSV_FILE' is not defined

: 